# 3. BERT y comparación

**Responsable: Sara.**

El extractor BERT y su prueba con diez textos proceden del cuaderno original
de Mónica. Aquí se ha añadido selección de GPU, guardado compatible con el cuaderno
de Izan, medición del truncamiento, comprobaciones de la prueba y registro de la
configuración. El fine-tuning y el análisis final están pendientes.

Ejecutar desde la carpeta del proyecto. Para la extracción completa conviene usar
Colab con GPU (Entorno de ejecución → Cambiar tipo de entorno → T4 GPU): en CPU tarda horas.

## Preparación en Colab

Solo hace falta en Colab. La carpeta del proyecto debe estar en Google Drive con
`funciones.py`, `resultados/particiones.csv` y los dos Excel en `datos/`. Trabajar
desde Drive conserva `datos/cache/bert.joblib` cuando se cierra la sesión.
Una forma sencilla de prepararla es clonar el repositorio desde Colab en
`/content/drive/MyDrive` y subir después los Excel a `datos/`.
Fuera de Colab esta celda no hace nada.

In [ ]:
import os
import sys

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RUTA_PROYECTO = "/content/drive/MyDrive/proyecto-cordis"  # ajustar si está en otra carpeta
    os.chdir(RUTA_PROYECTO)
    if RUTA_PROYECTO not in sys.path:
        sys.path.insert(0, RUTA_PROYECTO)
print("Carpeta de trabajo:", os.getcwd())

In [ ]:
import json
import platform
import time
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

from funciones import (
    RESULTADOS, CACHE, SEMILLA,
    guardar_representacion, cargar_representacion,
)

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
print("Python:", platform.python_version(),
      "| torch:", torch.__version__,
      "| transformers:", transformers.__version__)

In [ ]:
from funciones import cargar_datos, preparar_datos, cargar_o_crear_particiones

projects, scivoc = cargar_datos()
df = preparar_datos(projects, scivoc)
conjuntos = cargar_o_crear_particiones(df)  # comprueba IDs, etiquetas y orden con el CSV común
df_train = conjuntos["train"]
df_val = conjuntos["val"]
df_test = conjuntos["test"]
print({nombre: len(parte) for nombre, parte in conjuntos.items()})

## BERT congelado

Usamos el resumen original, sin el preprocesamiento de TF-IDF: BERT tiene su propio
tokenizador WordPiece y aprovecha mayúsculas, puntuación y palabras vacías como contexto
(el modelo *uncased* pasa a minúsculas internamente).

- **Modelo:** `bert-base-uncased` (12 capas, vectores de 768 dimensiones).
- **Congelado:** modo evaluación y sin gradientes. Sus pesos no cambian; solo se usa
  como extractor de características. Esto es lo que lo distingue del fine-tuning.
- **Un vector por documento:** media de los vectores de la última capa de todos los
  tokens reales. La máscara de atención excluye el padding. La media incluye `[CLS]` y
  `[SEP]`, como en la base inicial. Se prefiere la media al vector de `[CLS]` porque, sin
  fine-tuning, `[CLS]` no está entrenado para resumir el documento.
- **Textos largos:** se truncan a 512 tokens (el máximo de BERT, incluidos `[CLS]` y `[SEP]`).
  Más abajo se mide a cuántos resúmenes afecta.

Esta primera carga descarga los pesos si todavía no están en el ordenador.

In [ ]:
modelo_nombre = "bert-base-uncased"
MAX_LEN = 512
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(modelo_nombre)
modelo_bert = AutoModel.from_pretrained(modelo_nombre).to(dispositivo)
modelo_bert.requires_grad_(False)
modelo_bert.eval()
print("Dispositivo:", dispositivo,
      f"({torch.cuda.get_device_name(0)})" if dispositivo.type == "cuda" else "")

In [ ]:
textos = {
    nombre: parte["summary"].fillna("").astype(str).tolist()
    for nombre, parte in conjuntos.items()
}
for nombre, lista in textos.items():
    print(nombre, len(lista))

## ¿Cuántos resúmenes superan los 512 tokens?

Se tokeniza sin truncar para medir la longitud real en tokens WordPiece, contando
`[CLS]` y `[SEP]`. Es normal que aparezca el aviso *Token indices sequence length is
longer than the specified maximum*: aquí solo medimos y no se pasa nada al modelo.

In [ ]:
def longitudes_en_tokens(lista, tamano_lote=1000):
    longitudes = []
    for i in range(0, len(lista), tamano_lote):
        codificados = tokenizer(lista[i:i + tamano_lote], truncation=False)
        longitudes.extend(len(ids) for ids in codificados["input_ids"])
    return np.array(longitudes)

longitudes = {nombre: longitudes_en_tokens(lista) for nombre, lista in textos.items()}

filas = []
for nombre, L in longitudes.items():
    truncados = L > MAX_LEN
    filas.append(dict(
        conjunto=nombre,
        resumenes=len(L),
        tokens_media=round(L.mean(), 1),
        tokens_mediana=int(np.median(L)),
        tokens_p95=int(np.percentile(L, 95)),
        tokens_max=int(L.max()),
        truncados=int(truncados.sum()),
        porcentaje_truncados=round(100 * truncados.mean(), 2),
        tokens_perdidos_media=round(float((L[truncados] - MAX_LEN).mean()), 1) if truncados.any() else 0.0,
    ))
tabla_truncamiento = pd.DataFrame(filas)
display(tabla_truncamiento)

**Interpretación (completar con la tabla anterior):** si el porcentaje truncado es
pequeño, el truncamiento apenas afecta al experimento. Además, los resúmenes suelen
presentar el tema al principio. Una alternativa sería dividir los textos largos en
ventanas de 512 tokens y promediarlas, pero solo compensa si se truncan muchos resúmenes.

In [ ]:
def obtener_embeddings_bert(textos, batch_size=4, ordenar_por_longitud=True):
    """Devuelve una matriz (n, 768) en el MISMO orden que `textos`.

    Si ordenar_por_longitud=True, los lotes agrupan textos de longitud parecida para
    reducir el padding y acelerar la extracción. Cada vector se vuelve a escribir en
    su posición original, así que el orden de las filas no cambia.
    """
    n = len(textos)
    dimension = modelo_bert.config.hidden_size
    salida_total = np.zeros((n, dimension), dtype=np.float32)
    if n == 0:
        return salida_total
    orden = np.argsort([len(t) for t in textos], kind="stable") if ordenar_por_longitud else np.arange(n)

    for i in tqdm(range(0, n, batch_size), desc="Extrayendo BERT"):
        indices = orden[i:i + batch_size]
        entradas = tokenizer(
            [textos[j] for j in indices], padding=True, truncation=True,
            max_length=MAX_LEN, return_tensors="pt",
        )
        entradas = {k: v.to(dispositivo) for k, v in entradas.items()}
        with torch.no_grad():
            salida = modelo_bert(**entradas).last_hidden_state
            mascara = entradas["attention_mask"].unsqueeze(-1).to(salida.dtype)
            vector = (salida * mascara).sum(dim=1) / mascara.sum(dim=1).clamp(min=1)
        salida_total[indices] = vector.float().cpu().numpy()
    return salida_total

## Prueba con diez resúmenes

Antes de lanzar el corpus completo comprobamos:
1. que las dimensiones son (10, 768) y no hay valores no finitos;
2. que dos ejecuciones dan el mismo resultado;
3. que el padding no cambia el vector: procesar en lote o de uno en uno debe coincidir
   salvo ruido numérico (diferencias de 1e-4 o menos). Una diferencia grande indicaría
   que la máscara no está excluyendo el padding.

In [ ]:
muestra = textos["train"][:10]
X_prueba_bert = obtener_embeddings_bert(muestra, batch_size=5)
X_repetida = obtener_embeddings_bert(muestra, batch_size=5)
X_uno_a_uno = obtener_embeddings_bert(muestra, batch_size=1)

print("Dimensiones de la prueba:", X_prueba_bert.shape)
print("Todos los valores son finitos:", bool(np.isfinite(X_prueba_bert).all()))
print("Diferencia máxima entre dos ejecuciones:", float(np.abs(X_prueba_bert - X_repetida).max()))
print("Diferencia máxima lote vs. uno a uno:", float(np.abs(X_prueba_bert - X_uno_a_uno).max()))

## Extracción completa

Cambiar `GENERAR_TODO` a `True` cuando se quiera ejecutar el corpus completo.
Los vectores se guardan fuera de GitHub en `datos/cache/bert.joblib` y se recargan
para comprobar que coinciden. Izan podrá cargarlos con `cargar_representacion("bert", conjuntos)`;
hay que pasarle el archivo por otro medio (Drive, por ejemplo).

La configuración realmente ejecutada (modelo, dispositivo, lote, tiempos, versiones y
truncamiento) se guarda en `resultados/bert_congelado_config.json`, que sí va a GitHub.

In [ ]:
GENERAR_TODO = False

if GENERAR_TODO:
    batch_size = 32 if dispositivo.type == "cuda" else 4
    matrices_bert, minutos = {}, {}
    for nombre, lista in textos.items():
        inicio = time.time()
        matrices_bert[nombre] = obtener_embeddings_bert(lista, batch_size=batch_size)
        minutos[nombre] = round((time.time() - inicio) / 60, 2)
        assert matrices_bert[nombre].shape == (len(conjuntos[nombre]), modelo_bert.config.hidden_size)
        assert np.isfinite(matrices_bert[nombre]).all(), f"Valores no finitos en {nombre}"
        print(nombre, matrices_bert[nombre].shape, f"{minutos[nombre]} min")

    ruta = guardar_representacion("bert", matrices_bert, conjuntos)
    recargadas = cargar_representacion("bert", conjuntos)
    for nombre in matrices_bert:
        assert np.allclose(recargadas[nombre], matrices_bert[nombre]), nombre
    print("Vectores guardados y comprobados en:", ruta)

    registro = dict(
        fecha=datetime.now().isoformat(timespec="seconds"),
        modelo=modelo_nombre,
        congelado=True,
        pooling="media de la última capa sobre tokens con attention_mask=1 (incluye [CLS] y [SEP])",
        max_len=MAX_LEN,
        truncamiento="se conservan los primeros 512 tokens",
        texto="summary original, sin preprocesamiento",
        batch_size=batch_size,
        ordenado_por_longitud=True,
        dispositivo=torch.cuda.get_device_name(0) if dispositivo.type == "cuda" else "cpu",
        minutos=minutos,
        dimensiones={k: list(v.shape) for k, v in matrices_bert.items()},
        truncamiento_medido=tabla_truncamiento.to_dict(orient="records"),
        versiones=dict(python=platform.python_version(), torch=torch.__version__,
                       transformers=transformers.__version__),
        semilla=SEMILLA,
    )
    with open(RESULTADOS / "bert_congelado_config.json", "w", encoding="utf-8") as fichero:
        json.dump(registro, fichero, ensure_ascii=False, indent=2)
else:
    print("Extracción completa pendiente: activar GENERAR_TODO para ejecutarla.")

## Pendiente: fine-tuning

Preparar un Transformer para clasificación con Hugging Face y actualizar sus
pesos usando las etiquetas. Mantener el resumen original y las particiones.
Convertir los códigos de categoría a índices consecutivos y guardar el mapeo.
Anotar modelo, tokenización, longitud máxima, épocas, batch size, tasa de
aprendizaje y selección mediante validación. El test se usa al final.

No basta con entrenar una capa sobre los vectores congelados: este experimento
tiene que actualizar los parámetros del Transformer.


In [ ]:
# PENDIENTE: entrenamiento de clasificación con fine-tuning.


## Comparación de resultados disponibles

Esta tabla se rellena al ejecutar los experimentos. Separar validación y test,
nivel 1 y nivel 2; no mezclar resultados iniciales con los definitivos.


In [ ]:
metricas = pd.read_csv(RESULTADOS / "metricas.csv")
if metricas.empty:
    print("Todavía no hay métricas guardadas en esta copia del proyecto.")
else:
    display(metricas.sort_values(
        ["nivel", "conjunto", "macro_f1"], ascending=[True, True, False]
    ))


## Pendiente: evaluación final y nivel 1

Reunir resultados de Izan y del fine-tuning. Mostrar macro-F1, accuracy y métricas
por clase; incluir curvas de aprendizaje o validación y la matriz de confusión
completa de la mejor configuración.

Comparar el ranking de las tres representaciones en los niveles 1 y 2.
Si el fine-tuning es el mejor modelo global, incluir su repetición en nivel 1.


In [ ]:
# PENDIENTE: tablas y figuras de la evaluación final.


## Pendiente: incertidumbre y errores

Establecer qué diferencia de rendimiento es suficientemente clara con nuestro
test y aplicar ese criterio a las conclusiones. Usar predicciones de los
mismos proyectos para comparar modelos.

Revisar ejemplos, clases pequeñas y confusiones entre categorías hermanas.
Se puede comprobar si la predicción era otra de las categorías válidas del
proyecto que descartamos al elegir una sola etiqueta.


In [ ]:
# PENDIENTE: análisis de incertidumbre y ejemplos de errores.


## Para cerrar el trabajo

Integrar el texto que escriban Mónica e Izan en el README, incorporar todas las figuras
usadas en la discusión y revisar las conclusiones con el grupo.
Completar las contribuciones reales, las referencias y las diapositivas.
Comprobar que otra persona puede repetir el trabajo desde esta carpeta.
